In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmond 600 ~/.kaggle/kaggle.json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!kaggle datasets download -d asdasdasasdas/garbage-classification
!unzip garbage-classificaion.zip -d garbage_data

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from google.colab import drive

drive.mount('/content/drive')

train_dir = '/content/drive/MyDrive/60Pic'


train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(160, 160),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(160, 160),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 1544 images belonging to 31 classes.
Found 381 images belonging to 31 classes.


In [ ]:

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(160, 160, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(31, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(train_generator, epochs=20, validation_data=validation_generator)

Epoch 1/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 761s 15s/step - accuracy: 0.5576 - loss: 1.6996 - val_accuracy: 0.5538 - val_loss: 1.6566
Epoch 2/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 153s 3s/step - accuracy: 0.8517 - loss: 0.5215 - val_accuracy: 0.6693 - val_loss: 1.3055
Epoch 3/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 166s 3s/step - accuracy: 0.9080 - loss: 0.3350 - val_accuracy: 0.6588 - val_loss: 1.3446
Epoch 4/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 160s 3s/step - accuracy: 0.9372 - loss: 0.2184 - val_accuracy: 0.6798 - val_loss: 1.3156
Epoch 5/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 164s 3s/step - accuracy: 0.9391 - loss: 0.1966 - val_accuracy: 0.6850 - val_loss: 1.2375
Epoch 6/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 156s 3s/step - accuracy: 0.9566 - loss: 0.1571 - val_accuracy: 0.7218 - val_loss: 1.0652
Epoch 7/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 165s 3s/step - accuracy: 0.9566 - loss: 0.1306 - val_accuracy: 0.6903 - val_loss: 1.2019
Epoch 8/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 155s 3s/step - accuracy: 0.9637 - loss: 0.1252 - val_accuracy: 0.7612 - 

In [ ]:
model.save("face_recognition_model2.h5")
import json
labels = train_generator.class_indices
with open('labels2.json', 'w') as f:
    json.dump(labels, f)